# Alternative splicing from RNA-seq data

This mini-protocol produces normalized, gene-annotated splicing phenotypes for splicing-QTL analysis using LeafCutter.

#### Miniprotocol Timing

Timing: TBD

## Overview

The [splicing-calling module](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/calling/splicing_calling.html) quantifies splicing with LeafCutter, which derives intron excision ratios from STAR splice-junction files without relying on a transcript annotation.

The [normalization module](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/splicing_normalization.html) performs missingness and variability filtering followed by quantile normalization. The [gene-annotation module](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html) then assigns coordinates and phenotype groups required by TensorQTL.

## Steps

| **Analysis goal** | **Commands to run, in order** | **Inputs** |
| --- | --- | --- |
| LeafCutter from aligned RNA-seq data | 1 → 2 → 3 | `output/rnaseq/protocol_example.rnaseq.bam.list.txt`; STAR/WASP alignment directories |
| LeafCutter from a precomputed ratio matrix | 2 → 3 | `input/rnaseq/protocol_example.leafcutter.intron_usage_perind.counts.gz`; `tests/fixtures/gene_annotation/protocol_example.leafcutter.intron_count.tsv` |

Within the selected route, run the commands in numerical order. The bundled example data support the precomputed-matrix route; the calling route requires aligned BAM files.

### [1. Quantify intron usage with LeafCutter](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/calling/splicing_calling.html)

**What it does:** `leafcutter` extracts splice junctions and clusters introns to calculate per-sample intron excision ratios.

In [ ]:
sos run pipeline/splicing_calling.ipynb leafcutter   --cwd output/splicing/leafcutter   --samples output/rnaseq/protocol_example.rnaseq.bam.list.txt   --data-dir output/rnaseq/star_output_wasp

### [2. Normalize LeafCutter ratios](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/splicing_normalization.html)

**What it does:** `leafcutter_norm` filters introns and clusters, mean-imputes retained missing values, and quantile-normalizes the ratio matrix.

In [ ]:
sos run pipeline/splicing_normalization.ipynb leafcutter_norm   --cwd output/splicing/leafcutter   --ratios output/leafcutter/normalize/protocol_example.leafcutter.intron_usage_perind.counts.gz   --mean-impute

### [3. Annotate LeafCutter phenotypes](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html)

**What it does:** `annotate_leafcutter_isoforms` maps introns to genes and writes the coordinate-sorted phenotype matrix and phenotype-group file used by TensorQTL.

In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_leafcutter_isoforms   --cwd output/splicing/leafcutter   --phenoFile output/gene_annotation/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.qqnorm.txt   --intron-count tests/fixtures/gene_annotation/protocol_example.leafcutter.intron_count.tsv   --coordinate-annotation tests/fixtures/gene_annotation/Homo_sapiens.GRCh38.103.collapse_only.gene.chr22.gtf.gz   --map-stra site

## Output

| Route/step | Output filename and relative path | Description |
| --- | --- | --- |
| LeafCutter calling | `output/splicing/leafcutter/*_intron_usage_perind.counts.gz`; `*_intron_usage_perind_numers.counts.gz` | Intron excision ratios and supporting intron counts |
| LeafCutter normalization | `input/rnaseq/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.txt`; `input/rnaseq/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.qqnorm.txt` | QC-filtered and quantile-normalized ratios |
| LeafCutter annotation | `output/splicing/leafcutter/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.qqnorm.formated.bed.gz`; `output/splicing/leafcutter/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.qqnorm.phenotype_group.txt` | TensorQTL phenotype and grouping files |

## Anticipated Results

Each route ends with a coordinate-sorted, bgzip-compressed phenotype matrix and a matching phenotype-group file. Rows represent introns, columns represent samples, and the normalized values can be supplied directly to splicing-QTL association testing.

## Command Interface

In [ ]:
sos run pipeline/splicing_calling.ipynb -h

In [ ]:
sos run pipeline/splicing_normalization.ipynb -h

In [ ]:
sos run pipeline/gene_annotation.ipynb -h